# Почему ROC-AUC "врет" при дисбалансе классов?

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import (auc, average_precision_score,
                             precision_recall_curve, roc_curve)

In [2]:
def plot_distribution(y_true, y_score, model_name):
    df = pd.DataFrame({"class": y_true, "score": y_score})
    fig = px.histogram(
        df,
        x="score",
        color="class",
        nbins=50,
        title=f"{model_name}: распределение скоров",
        labels={"score": "Скор", "count": "Количество", "class": "Класс"},
        barmode="overlay",
        opacity=0.6,
    )
    fig.update_layout(legend=dict(title="Истинный класс"))
    return fig

In [3]:
def plot_roc_curve(y_true, y_score, model_name):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(x=fpr, y=tpr, mode="lines", name=f"ROC (AUC={roc_auc:.3f})")
    )
    fig.add_trace(
        go.Scatter(
            x=[0, 1],
            y=[0, 1],
            mode="lines",
            name="Случайная модель",
            line=dict(dash="dash"),
        )
    )
    fig.update_layout(
        title=f"{model_name}: ROC-curve",
        xaxis_title="False Positive Rate",
        yaxis_title="True Positive Rate",
    )
    return fig

In [4]:
def plot_pr_curve(y_true, y_score, model_name):
    precision, recall, _ = precision_recall_curve(y_true, y_score)
    ap = average_precision_score(y_true, y_score)
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=recall,
            y=precision,
            mode="lines",
        )
    )
    fig.update_layout(
        title=f"{model_name}: Precision-Recall curve | PR-AUC={ap:.3f}",
        xaxis_title="Recall",
        yaxis_title="Precision",
    )
    return fig

In [5]:
def create_model_df(y_true, y_score):
    df = pd.DataFrame(
        {
            "true_class": y_true,
            "score": y_score.round(2),
        }
    )
    df = df.sort_values(by="score", ascending=False).reset_index(drop=True)
    return df

In [6]:
y_true_A = np.array([1]*10 + [0]*90)
pos_scores_A = np.linspace(0.95, 0.86, 10)
neg_scores_A = np.linspace(0.85, 0.01, 90)
y_score_A = np.concatenate([pos_scores_A, neg_scores_A])

y_true_B = np.array([1]*10 + [0]*90)
pos_scores_B = [0.99] + list(np.linspace(0.40, 0.32, 9))
neg_scores_B = list(np.linspace(0.90, 0.41, 50)) + list(np.linspace(0.39, 0.01, 40))
y_score_B = np.array(pos_scores_B + neg_scores_B)

y_true_C = np.array([1]*10 + [0]*90)
pos_scores_C = [0.99] + list(np.linspace(0.80, 0.72, 9))
neg_scores_C = list(np.linspace(0.90, 0.71, 20)) + list(np.linspace(0.70, 0.01, 70))
y_score_C = np.array(pos_scores_C + neg_scores_C)

In [7]:
df_A = create_model_df(y_true_A, y_score_A)
df_B = create_model_df(y_true_B, y_score_B)
df_C = create_model_df(y_true_C, y_score_C)

## Модель A: идеальное разделение

In [8]:
df_A

,true_class,score
0,1,0.95
1,1,0.94
2,1,0.93
3,1,0.92
4,1,0.91
...,...,...
95,0,0.05
96,0,0.04
97,0,0.03
98,0,0.02


In [9]:
fig_dist_A = plot_distribution(y_true_A, y_score_A, "Модель A")
fig_dist_A.show()

fig_roc_A = plot_roc_curve(y_true_A, y_score_A, "Модель A")
fig_roc_A.show()

fig_pr_A = plot_pr_curve(y_true_A, y_score_A, "Модель A")
fig_pr_A.show()

## Модель B: плохая (ROC-AUC ~0.5)

In [10]:
df_B

,true_class,score
0,1,0.99
1,0,0.90
2,0,0.89
3,0,0.88
4,0,0.87
...,...,...
95,0,0.05
96,0,0.04
97,0,0.03
98,0,0.02


In [11]:
fig_dist_B = plot_distribution(y_true_B, y_score_B, "Модель B")
fig_dist_B.show()

fig_roc_B = plot_roc_curve(y_true_B, y_score_B, "Модель B")
fig_roc_B.show()

fig_pr_B = plot_pr_curve(y_true_B, y_score_B, "Модель B")
fig_pr_B.show()

## Модель C: обманчивая (высокий ROC-AUC)

In [12]:
df_C

,true_class,score
0,1,0.99
1,0,0.90
2,0,0.89
3,0,0.88
4,0,0.87
...,...,...
95,0,0.05
96,0,0.04
97,0,0.03
98,0,0.02


In [13]:
fig_dist_C = plot_distribution(y_true_C, y_score_C, "Модель C")
fig_dist_C.show()

fig_roc_C = plot_roc_curve(y_true_C, y_score_C, "Модель C")
fig_roc_C.show()

fig_pr_C = plot_pr_curve(y_true_C, y_score_C, "Модель C")
fig_pr_C.show()

# Вывод
**Интерпретация:**
- Модель A: идеальное разделение -> ROC-AUC = 1.0, PR-AUC = 1.0.
- Модель B: положительные в основном ниже многих отрицательных -> ROC-AUC ≈ 0.5 (случайная), PR-AUC низкий.
- Модель C: один положительный очень высоко, остальные выше половины отрицательных -> ROC-AUC ≈ 0.8 (высокий!), но PR-AUC значительно ниже (≈0.2), что отражает плохое качество поиска положительного класса.

**Вывод:** при дисбалансе классов ROC-AUC может давать завышенную оценку. Используйте PR-AUC для более реалистичной картины.